# In Summary
model-3 uses a frame stack of 3. It does not make sense to make use of a frame stack in the game of WASD, since the action should be determined on a single frame. There is no other relevant information in previous frames which have to be taken into account. But it was a fun experiment to try either way.

installs
- gymnasium: 0.29.1
- pygame
- matplotlib: 3.9.2
- torch: 2.4.1+cpu
- torchvision: 0.19.1+cpu
- torchaudio: 2.4.1+cpu
- torchsummary

In [93]:
import gymnasium as gym
import matplotlib
import torch
import torchvision
import torchaudio
import torchsummary

print(gym.__version__)          # 0.29.1
print(matplotlib.__version__)   # 3.9.2
print(torch.__version__)        # 2.4.1
print(torchvision.__version__)  # 0.19.1
print(torchaudio.__version__)   # 2.4.1

0.29.1
3.9.2
2.4.1+cpu
0.19.1+cpu
2.4.1+cpu


1. Setup the CNN for Q-Learning

In [94]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
from torchvision import transforms

STACK_SIZE = 3

# Define the CNN Q-network
class DQNCNN(nn.Module):
    def __init__(self, action_size):
        super(DQNCNN, self).__init__()
        self.conv1 = nn.Conv2d(STACK_SIZE, 64, kernel_size=3, stride=2)  # Using default stride and padding
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Reduces size further

        # Adjust this size based on your calculations
        self.fc1 = nn.Linear(64 * 20 * 20, 512)
        self.fc2 = nn.Linear(512, action_size)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)  # Apply max pooling to reduce size further
        x = x.view(x.size(0), -1)  # Flatten the tensor
        x = torch.relu(self.fc1(x))
        return self.fc2(x)


2. Preprocessing Function

In [95]:
# Preprocess the frames (resize and convert to grayscale)
def preprocess_frame(frame):
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Grayscale(),  # Convert to grayscale
        transforms.Resize((84, 84)),  # Resize to 84x84
        transforms.ToTensor(),  # Convert to tensor, will have shape [1, 84, 84]
    ])
    return transform(frame).squeeze(0)  # Remove the single channel dimension, resulting in [84, 84]


# Stack 4 frames
def stack_frames(frames):
    return np.stack(frames, axis=0)  # Shape will be [4, 84, 84]


3. Experience Replay Buffer

In [96]:
# Replay buffer to store past experiences
class ReplayBuffer:
    def __init__(self, buffer_size=10000):
        self.memory = deque(maxlen=buffer_size)

    def add(self, experience):
        self.memory.append(experience)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)


4. DQN Agent

In [97]:
class DQNAgent:
    def __init__(self, action_size):
        self.action_size = action_size
        self.memory = ReplayBuffer(buffer_size=10000) 
        self.gamma = 0.99  # Discount factor
        self.epsilon = 0.5  # Exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 64
        self.update_frequency = 4

        # Create two networks: one for the current Q-function and one for the target Q-function
        self.q_network = DQNCNN(action_size)
        self.target_network = DQNCNN(action_size)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=0.0001)

        # Copy weights from the current network to the target network
        self.target_network.load_state_dict(self.q_network.state_dict())

    def select_action(self, state, explore=True):
        # During exploration, select a random action
        if explore and np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        
        # During exploitation, select the action with the highest Q-value
        state = torch.FloatTensor(state).unsqueeze(0)  # Convert to tensor and add batch dimension
        q_values = self.q_network(state)
        return torch.argmax(q_values).item()

    def replay(self):
        if len(self.memory) < self.batch_size:
            return

        minibatch = self.memory.sample(self.batch_size)
        states, actions, rewards, next_states, dones = zip(*minibatch)

        # Convert to numpy arrays before converting to tensors
        states = torch.FloatTensor(np.array(states))
        next_states = torch.FloatTensor(np.array(next_states))
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        dones = torch.FloatTensor(dones)

        # Get current Q values
        q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # Get target Q values
        next_q_values = self.target_network(next_states).max(1)[0]
        target_q_values = rewards + (self.gamma * next_q_values * (1 - dones))

        # Compute loss
        loss = nn.MSELoss()(q_values, target_q_values.detach())
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Update epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target_network(self):
        self.target_network.load_state_dict(self.q_network.state_dict())


5. Initialise Environment

In [98]:
from WASDEnv import WinFormsGameEnv

# env = gym.make('CartPole-v1', render_mode='rgb_array')
env = WinFormsGameEnv()

6. Initialise Agent

In [99]:
agent = DQNAgent(action_size=env.action_space.n)

In [100]:
agent

7. Training Loop

In [88]:
# Stack size for CNN input (4 stacked frames)
stack_size = STACK_SIZE
agent.q_network.train()

# Training loop
num_episodes = 1000
for e in range(num_episodes):
    state_stack = deque(maxlen=stack_size)
    state, info = env.reset()
    
    # Initialize frame stack with initial frame
    frame = preprocess_frame(env.render())
    for _ in range(stack_size):
        state_stack.append(frame)

    done = False
    total_reward = 0
    while not done:
        env.render()
        state = stack_frames(state_stack)

        # Choose an action
        action = agent.select_action(state)
        
        # Perform action
        next_state, reward, done, _, _ = env.step(action)
        
        # Process next state (frame) and add to the stack
        next_frame = preprocess_frame(env.render())
        state_stack.append(next_frame)
        next_state_stack = stack_frames(state_stack)

        # Store experience in replay memory
        agent.memory.add((state, action, reward, next_state_stack, done))
        
        # Train the agent with experience replay
        agent.replay()

        # Update the state
        state_stack = state_stack

        total_reward += reward
        
        # Periodically update target network
        if e % agent.update_frequency == 0:
            agent.update_target_network()

    print(f"Episode: {e+1}/{num_episodes}, Total Reward: {total_reward}")

env.close()


Episode: 1/1000, Total Reward: 0
Episode: 2/1000, Total Reward: 1
Episode: 3/1000, Total Reward: 0
Episode: 4/1000, Total Reward: 0
Episode: 5/1000, Total Reward: 0
Episode: 6/1000, Total Reward: 0
Episode: 7/1000, Total Reward: 1
Episode: 8/1000, Total Reward: 0
Episode: 9/1000, Total Reward: 0
Episode: 10/1000, Total Reward: 1
Episode: 11/1000, Total Reward: 0
Episode: 12/1000, Total Reward: 0
Episode: 13/1000, Total Reward: 1
Episode: 14/1000, Total Reward: 0
Episode: 15/1000, Total Reward: 1
Episode: 16/1000, Total Reward: 0
Episode: 17/1000, Total Reward: 0
Episode: 18/1000, Total Reward: 0
Episode: 19/1000, Total Reward: 0
Episode: 20/1000, Total Reward: 0
Episode: 21/1000, Total Reward: 0
Episode: 22/1000, Total Reward: 0
Episode: 23/1000, Total Reward: 0
Episode: 24/1000, Total Reward: 0
Episode: 25/1000, Total Reward: 0
Episode: 26/1000, Total Reward: 0
Episode: 27/1000, Total Reward: 0
Episode: 28/1000, Total Reward: 0
Episode: 29/1000, Total Reward: 0
Episode: 30/1000, Total

error: (1400, 'SetForegroundWindow', 'Invalid window handle.')

8. Save Model Weights

In [90]:
import torch

# Assuming `agent` is your trained model and `path_to_save` is the file path
torch.save(agent.q_network.state_dict(), './model/q_network_200.pth')
torch.save(agent.target_network.state_dict(), './model/target_network_200.pth')

9. Load Model Weights

In [101]:
import torch

# Assuming `Agent` is your model class
agent.q_network = DQNCNN(env.action_space.n)  # Initialize the model architecture
agent.q_network.load_state_dict(torch.load('./model-3/q_network_200.pth'))

agent.target_network = DQNCNN(env.action_space.n)  # Initialize the model architecture
agent.target_network.load_state_dict(torch.load('./model-3/target_network_200.pth'))

# Ensure the models are in evaluation mode
agent.q_network.eval()
agent.target_network.eval()


C:\Users\Kaan\AppData\Local\Temp\ipykernel_20268\1099591358.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  agent.q_network.load_state_dict(torch.load('./model-3/q_netwo

DQNCNN(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=25600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=4, bias=True)
)

10. Testing Loop

In [102]:
import matplotlib.pyplot as plt 

# Testing loop (after training)
num_test_episodes = 3
stack_size = STACK_SIZE
all_rewards = 0

for e in range(num_test_episodes):
    state_stack = deque(maxlen=stack_size)
    state, info = env.reset()

    # Initialize frame stack with the first frame
    frame = preprocess_frame(env.render())
    for _ in range(stack_size):
        state_stack.append(frame)

    done = False
    total_reward = 0

    while not done:
        # Render the frame
        # clear_output(wait=True)
        # plt.imshow(env.render())
        # plt.axis('off')  # Turn off the axis

        state = stack_frames(state_stack)

        # Use the trained model to select an action (no exploration during testing)
        action = agent.select_action(state, explore=False)  # No exploration in testing

        # Perform the action in the environment
        next_state, reward, done, _, _ = env.step(action)

        # Process the next state (frame) and update the frame stack
        next_frame = preprocess_frame(env.render())
        state_stack.append(next_frame)

        total_reward += reward

        # Update the plot with the new frame
        # plt.title(f"Test Episode: {e + 1}/{num_test_episodes}, Total Reward: {total_reward}")
        # plt.pause(0.001)  # Short pause to allow plot to update

    print(f"Test Episode: {e + 1}/{num_test_episodes}, Total Reward: {total_reward}")
    all_rewards += total_reward
print(f'avg score: {all_rewards/num_test_episodes}')
env.close()


KeyboardInterrupt: 

In [61]:
from IPython.display import clear_output
import matplotlib.pyplot as plt
from PIL import Image
import random

# Testing loop (after training)
num_test_episodes = 100
stack_size = 4
all_rewards = 0

for e in range(num_test_episodes):
    state_stack = deque(maxlen=stack_size)
    state, info = env.reset()

    # Initialize frame stack with the first frame
    frame = preprocess_frame(env.render())
    for _ in range(stack_size):
        state_stack.append(frame)

    done = False
    total_reward = 0

    while not done:
        # Render the frame
        # clear_output(wait=True)
        # plt.imshow(env.render())
        # plt.axis('off')  # Turn off the axis

        state = stack_frames(state_stack)

        # Use the trained model to select an action (no exploration during testing)
        action = random.choice([0, 1])

        # Perform the action in the environment
        next_state, reward, done, _, _ = env.step(action)

        # Process the next state (frame) and update the frame stack
        next_frame = preprocess_frame(env.render())
        state_stack.append(next_frame)

        total_reward += reward

        # Update the plot with the new frame
        #plt.title(f"Test Episode: {e + 1}/{num_test_episodes}, Total Reward: {total_reward}")
        #plt.pause(0.001)  # Short pause to allow plot to update

    print(f"Test Episode: {e + 1}/{num_test_episodes}, Total Reward: {total_reward}")
    all_rewards += total_reward
print(f'avg score: {all_rewards/num_test_episodes}')

env.close()


Test Episode: 1/100, Total Reward: 18.0
Test Episode: 2/100, Total Reward: 37.0
Test Episode: 3/100, Total Reward: 16.0
Test Episode: 4/100, Total Reward: 11.0
Test Episode: 5/100, Total Reward: 61.0
Test Episode: 6/100, Total Reward: 13.0
Test Episode: 7/100, Total Reward: 26.0
Test Episode: 8/100, Total Reward: 32.0
Test Episode: 9/100, Total Reward: 23.0
Test Episode: 10/100, Total Reward: 14.0
Test Episode: 11/100, Total Reward: 38.0
Test Episode: 12/100, Total Reward: 14.0
Test Episode: 13/100, Total Reward: 42.0
Test Episode: 14/100, Total Reward: 33.0
Test Episode: 15/100, Total Reward: 12.0
Test Episode: 16/100, Total Reward: 19.0
Test Episode: 17/100, Total Reward: 83.0
Test Episode: 18/100, Total Reward: 41.0
Test Episode: 19/100, Total Reward: 12.0
Test Episode: 20/100, Total Reward: 11.0
Test Episode: 21/100, Total Reward: 18.0
Test Episode: 22/100, Total Reward: 31.0
Test Episode: 23/100, Total Reward: 13.0
Test Episode: 24/100, Total Reward: 12.0
Test Episode: 25/100, Tot

11. Visualise Model

In [32]:
from torchsummary import summary

# Assuming your model is named 'model' and your input size is (channels, height, width)
summary(agent.q_network, input_size=(4, 84, 84))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 20, 20]           8,224
            Conv2d-2             [-1, 64, 9, 9]          32,832
            Conv2d-3             [-1, 64, 7, 7]          36,928
            Linear-4                  [-1, 512]       1,606,144
            Linear-5                    [-1, 2]           1,026
Total params: 1,685,154
Trainable params: 1,685,154
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.11
Forward/backward pass size (MB): 0.17
Params size (MB): 6.43
Estimated Total Size (MB): 6.70
----------------------------------------------------------------


# Results
- No Max Pooling (17250): 40
- With Max Pooling (1600): 103
- With Max Pooling (1950): 213
- With Max Pooling (2075): 200
- With Max Pooling (2150): 190
- With Max Pooling (2225): 202
- With Max Pooling (1950) (2): 205


Max pooling speeds up the process and helps the model understand the case better